### Part 3: PageRank on Spark

- **Dataset:** `small.txt` (53 nodes) and `whole.txt` (1000 nodes, 8192 edges)  
- **Goal:** Implement iterative PageRank using Spark RDDs  
- **Parameters:** 40 iterations, β = 0.8, teleport probability = 1 - β = 0.2  
- **Verify:** Top score on small.txt should be ~0.036

In [2]:
import os
from pyspark import SparkContext

# Stop any existing context to avoid conflicts when re-running
try:
    sc.stop()
except:
    pass

sc = SparkContext(appName="Part3_PageRank")
sc.setLogLevel("ERROR")  # only show errors, not info/warn spam

SMALL_GRAPH = "Q3_PageRank/small.txt"
WHOLE_GRAPH = "Q3_PageRank/whole.txt"

print("Spark started, version:", sc.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 19:34:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started, version: 4.1.1


### Dataset Details

- Each line in the file represents an edge: `source_node  destination_node`
- Duplicate edges may exist — treat them as a single edge
- The graph includes a directed cycle of 1000 edges, ensuring it is strongly connected

In [3]:
def load_graph(filepath):
    """
    Load a graph from a text file.
    Each line: 'src dst'
    Returns:
      edges     : RDD of (src, dst) tuples, duplicates removed
      all_nodes : RDD of all unique node IDs
      n         : total number of nodes
    """
    raw = sc.textFile(filepath)

    # Parse each line into (src, dst) integer pairs
    # distinct() removes duplicate edges as instructed
    edges = raw \
        .map(lambda line: line.strip().split()) \
        .filter(lambda parts: len(parts) == 2) \
        .map(lambda parts: (int(parts[0]), int(parts[1]))) \
        .distinct()

    # Collect all unique node IDs (from both source and destination columns)
    all_nodes = edges \
        .flatMap(lambda e: [e[0], e[1]]) \
        .distinct()

    n = all_nodes.count()
    return edges, all_nodes, n

# Test on the small graph first
edges_small, nodes_small, n_small = load_graph(SMALL_GRAPH)

print(f"Small graph:")
print(f"  Nodes : {n_small}")
print(f"  Edges : {edges_small.count()}")
print(f"  Sample edges: {edges_small.take(5)}")

Small graph:
  Nodes : 100
  Edges : 950
  Sample edges: [(13, 1), (89, 1), (79, 1), (65, 1), (25, 1)]


### Build Transition Matrix (RDD)

- Construct the transition matrix **M** as an RDD  
- \( M[i][j] = \frac{1}{\deg(i)} \) if edge \( i \rightarrow j \) exists, otherwise 0  
- \( \deg(i) \) is the out-degree of node \( i \) (ignoring duplicate edges)  
- Represent **M** as an RDD of `(src, dst, weight)` where `weight = 1 / deg(src)`  
- Matrix is sparse — only non-zero entries are stored  

In [4]:
def build_transition(edges):
    """
    Build sparse transition matrix from edges RDD.
    Returns RDD of (dst, (src, weight)) — grouped by destination
    so we can compute M*r efficiently.
    """
    # Count out-degree of each source node
    out_degree = edges \
        .map(lambda e: (e[0], 1)) \
        .reduceByKey(lambda a, b: a + b)
    # out_degree: RDD of (src, degree)

    # Join edges with their source node's out-degree
    # Then compute the transition weight = 1 / degree
    # Result: (dst, (src, 1/deg(src)))
    transition = edges \
        .map(lambda e: (e[0], e[1])) \
        .join(out_degree) \
        .map(lambda x: (x[1][0], (x[0], 1.0 / x[1][1])))
    #                   dst        src   weight

    return transition, out_degree

transition_small, out_deg_small = build_transition(edges_small)
print("Transition matrix built.")
print("Sample entries (dst, (src, weight)):", transition_small.take(3))

Transition matrix built.
Sample entries (dst, (src, weight)): [(6, (24, 0.1111111111111111)), (50, (24, 0.1111111111111111)), (66, (24, 0.1111111111111111))]


### PageRank Iterative Algorithm

- Formula: \( r_{new} = \frac{1 - \beta}{n} \cdot [1,1,\dots,1] + \beta \cdot M \cdot r_{old} \)  
- First term is **teleportation**: with probability \( (1 - \beta) \), jump uniformly to any page  
- Ensures every page gets a non-zero rank  
- Second term: with probability \( \beta \), follow outgoing links using matrix \( M \)  

In [5]:
def pagerank(filepath, n_iter=40, beta=0.8):
    """
    Run PageRank on the graph in 'filepath'.
    Returns sorted list of (node_id, rank_score) — highest rank first.
    """
    # Step 1: Load the graph
    edges, all_nodes, n = load_graph(filepath)
    print(f"  Graph loaded: {n} nodes")

    # Step 2: Build transition matrix
    transition, out_degree = build_transition(edges)

    # Persist transition matrix — we reuse it every iteration
    transition.cache()

    # Step 3: Initialize rank vector r = 1/n for all nodes
    # r is an RDD of (node_id, rank)
    r = all_nodes.map(lambda node: (node, 1.0 / n))

    teleport_contrib = (1.0 - beta) / n  # constant teleport term added each iteration

    # Step 4: Iterate
    for i in range(n_iter):
        # Compute beta * M * r:
        #   For each edge (src -> dst) with weight w,
        #   dst receives: beta * w * r[src]
        #
        # transition RDD has: (dst, (src, weight))
        # r RDD has:          (node, rank)
        # Join on src:
        #   transition.map -> (src, (dst, weight))
        #   join r         -> (src, ((dst, weight), rank))
        #   map            -> (dst, beta * weight * rank)

        contributions = transition \
            .map(lambda x: (x[1][0], (x[0], x[1][1]))) \
            .join(r) \
            .map(lambda x: (x[1][0][0], beta * x[1][0][1] * x[1][1]))
        #                    dst         beta * weight     * rank

        # Sum up contributions arriving at each node
        summed = contributions.reduceByKey(lambda a, b: a + b)

        # Add teleport term to every node
        # Nodes with no incoming links still get the teleport contribution
        r = all_nodes \
            .map(lambda node: (node, 0.0)) \
            .union(summed) \
            .reduceByKey(lambda a, b: a + b) \
            .mapValues(lambda v: v + teleport_contrib)

        # Persist r every 10 iterations to avoid recomputing long lineages
        if (i + 1) % 10 == 0:
            r.cache()
            print(f"  Iteration {i+1}/{n_iter} done")

    # Step 5: Collect and sort results
    results = r.collect()
    results.sort(key=lambda x: -x[1])  # sort descending by rank score
    return results

print("pagerank() function defined.")

pagerank() function defined.


### Run on Small Graph

- Running PageRank on `small.txt` first  


In [6]:
print("Running PageRank on small.txt...")
print("=" * 55)

results_small = pagerank(SMALL_GRAPH, n_iter=40, beta=0.8)

print("\nTop 5 nodes (highest PageRank):")
for rank, (node, score) in enumerate(results_small[:5], 1):
    print(f"  #{rank}  Node {node:>5d}  score = {score:.6f}")

print("\nBottom 5 nodes (lowest PageRank):")
for rank, (node, score) in enumerate(results_small[-5:], 1):
    print(f"  #{rank}  Node {node:>5d}  score = {score:.6f}")

top_score = results_small[0][1]
print(f"\nTop score: {top_score:.6f}")
if abs(top_score - 0.036) < 0.005:
    print("Verification PASSED — top score is close to expected 0.036")
else:
    print("Verification note: score differs from 0.036, check beta and n_iter values")

Running PageRank on small.txt...
  Graph loaded: 100 nodes
  Iteration 10/40 done
  Iteration 20/40 done
  Iteration 30/40 done
  Iteration 40/40 done



Top 5 nodes (highest PageRank):
  #1  Node    53  score = 0.035731
  #2  Node    14  score = 0.034171
  #3  Node    40  score = 0.033630
  #4  Node     1  score = 0.030006
  #5  Node    27  score = 0.029720

Bottom 5 nodes (lowest PageRank):
  #1  Node    89  score = 0.003922
  #2  Node    37  score = 0.003808
  #3  Node    81  score = 0.003695
  #4  Node    59  score = 0.003670
  #5  Node    85  score = 0.003410

Top score: 0.035731
Verification PASSED — top score is close to expected 0.036


### Run on Full Graph

- Run PageRank on `whole.txt`  
- Graph size: 1000 nodes, 8192 edges  

In [7]:
import time

print("Running PageRank on whole.txt (1000 nodes)...")
print("This may take a few minutes on a laptop.")
print("=" * 55)

t_start = time.time()
results_whole = pagerank(WHOLE_GRAPH, n_iter=40, beta=0.8)
elapsed = time.time() - t_start

print(f"\nCompleted in {elapsed:.1f} seconds")

print("\nTop 5 nodes (highest PageRank):")
for rank, (node, score) in enumerate(results_whole[:5], 1):
    print(f"  #{rank}  Node {node:>5d}  score = {score:.6f}")

print("\nBottom 5 nodes (lowest PageRank):")
for rank, (node, score) in enumerate(results_whole[-5:], 1):
    print(f"  #{rank}  Node {node:>5d}  score = {score:.6f}")

Running PageRank on whole.txt (1000 nodes)...
This may take a few minutes on a laptop.
  Graph loaded: 1000 nodes
  Iteration 10/40 done
  Iteration 20/40 done
  Iteration 30/40 done
  Iteration 40/40 done



Completed in 68.0 seconds

Top 5 nodes (highest PageRank):
  #1  Node   263  score = 0.002020
  #2  Node   537  score = 0.001943
  #3  Node   965  score = 0.001925
  #4  Node   243  score = 0.001853
  #5  Node   285  score = 0.001827

Bottom 5 nodes (lowest PageRank):
  #1  Node   408  score = 0.000388
  #2  Node   424  score = 0.000355
  #3  Node    62  score = 0.000353
  #4  Node    93  score = 0.000351
  #5  Node   558  score = 0.000329


### Analysis — Rank Distribution

- Nodes with many incoming links receive higher PageRank  
- Well-connected nodes in the cycle tend to dominate the ranking  
- Nodes with few incoming links rely mainly on the teleport term  
- Such nodes have ranks close to \( 1/n \)  

In [8]:
scores = [score for _, score in results_whole]
n      = len(scores)
uniform = 1.0 / n  # what rank would be if perfectly uniform

print(f"Total nodes          : {n}")
print(f"Uniform rank (1/n)   : {uniform:.6f}")
print(f"Max rank             : {max(scores):.6f}  ({max(scores)/uniform:.1f}x uniform)")
print(f"Min rank             : {min(scores):.6f}  ({min(scores)/uniform:.2f}x uniform)")
print(f"Average rank         : {sum(scores)/n:.6f}")
print()
print("Observation: nodes in the directed cycle accumulate higher rank")
print("because they receive links from many other nodes.")

Total nodes          : 1000
Uniform rank (1/n)   : 0.001000
Max rank             : 0.002020  (2.0x uniform)
Min rank             : 0.000329  (0.33x uniform)
Average rank         : 0.001000

Observation: nodes in the directed cycle accumulate higher rank
because they receive links from many other nodes.


In [9]:
# Cleaning up Spark
sc.stop()
print("Spark context stopped")

Spark context stopped
